## 12b - Final Test Evaluation

This notebook performs the final test evaluation of the models selected during the validation stage:

- Demographics + Questionnaire - Logistic Regression
- Wearable + Questionnaire - Random Forest
- Full Multimodal - XGBoost 

The saved fitted pipelines are loaded directly and applied to the test data without refitting or retuning.

#### Objective

The objective of this notebook is to assess the final generalization performance of the validation-selected models on unseen test data and compare their test performance with the results obtained during validation.

The notebook will:

- Load the models selected during validation.
- Evaluate each selected model on its corresponding held-out test dataset.
- Calculate final performance metrics, including Accuracy, Balanced Accuracy, Macro F1, Macro - - Precision, and Macro Recall.
- Generate final confusion matrices.
- Compare validation and test performance for each selected model.

#### 1. Setup

In [6]:
from pathlib import Path
import os
import sys
import re
import warnings

os.environ.pop("MPLBACKEND", None)

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# ==========================================================
# Locate project root
# ==========================================================

cwd = Path.cwd().resolve()

if (cwd / "src").exists():
    PROJECT_ROOT = cwd

elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent

else:
    raise FileNotFoundError(
        "Could not locate the project root."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

# ========================================================
# Import shared evaluation utilities
# ========================================================

from src.modeling.evaluation import (
    calculate_metrics,
    get_classification_report,
    get_confusion_matrix,
)

# =======================================================
# Project directories
# =======================================================

DATA_DIR = (
    PROJECT_ROOT
    / "data"
)

PROCESSED_DIR = (
    DATA_DIR
    / "processed"
)

MODELS_DIR = (
    PROJECT_ROOT
    / "models"
)

OUTPUTS_DIR = (
    PROJECT_ROOT
    / "outputs"
)

METRICS_DIR = (
    OUTPUTS_DIR
    / "metrics"
)

TABLES_DIR = (
    OUTPUTS_DIR
    / "tables"
)

FIGURES_DIR = (
    OUTPUTS_DIR
    / "figures"
)

for directory in [
    METRICS_DIR,
    TABLES_DIR,
    FIGURES_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

# =======================================================
# Shared columns
# =======================================================

TARGET = "label"
ID_COLUMN = "patient_id"

print("Project root:", PROJECT_ROOT)
print("Models directory:", MODELS_DIR)
print("Processed data:", PROCESSED_DIR)
print("Metrics directory:", METRICS_DIR)
print("Tables directory:", TABLES_DIR)

Project root: C:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease
Models directory: C:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\models
Processed data: C:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\data\processed
Metrics directory: C:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\outputs\metrics
Tables directory: C:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\outputs\tables


#### 2. Load Validation-Selected Models

In [7]:
VALIDATION_BEST_MODELS_FILE = (
    TABLES_DIR
    / "validation_best_models.csv"
)

VALIDATION_RESULTS_FILE = (
    METRICS_DIR
    / "validation_results.csv"
)

if not VALIDATION_BEST_MODELS_FILE.exists():
    raise FileNotFoundError(
        f"Missing validation-selected model file: "
        f"{VALIDATION_BEST_MODELS_FILE}"
    )

if not VALIDATION_RESULTS_FILE.exists():
    raise FileNotFoundError(
        f"Missing validation results file: "
        f"{VALIDATION_RESULTS_FILE}"
    )

selected_candidates = pd.read_csv(
    VALIDATION_BEST_MODELS_FILE
)

validation_results = pd.read_csv(
    VALIDATION_RESULTS_FILE
)

print(
    "Validation-selected candidates:",
    len(selected_candidates)
)

display(
    selected_candidates[
        [
            "dataset",
            "model",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "precision_macro",
            "recall_macro",
        ]
    ].round(4)
)

Validation-selected candidates: 3


,dataset,model,accuracy,balanced_accuracy,macro_f1,precision_macro,recall_macro
0,Demographics + Questionnaire,Logistic Regression,0.6571,0.7016,0.6329,0.6183,0.7016
1,Full Multimodal,XGBoost,0.7714,0.6995,0.7071,0.7292,0.6995
2,Wearable + Questionnaire,Random Forest,0.8000,0.7862,0.7650,0.7589,0.7862


#### 3. Define and Verify Final Test Inputs

In [8]:
MODEL_FILES = {
    (
        "Demographics + Questionnaire",
        "Logistic Regression",
    ):
        MODELS_DIR
        / "demographics_plus_questionnaire_logistic_regression.joblib",

    (
        "Wearable + Questionnaire",
        "Random Forest",
    ):
        MODELS_DIR
        / "wearable_plus_questionnaire_random_forest.joblib",

    (
        "Full Multimodal",
        "XGBoost",
    ):
        MODELS_DIR
        / "full_multimodal_xgboost.joblib",
}

In [9]:
# =============================================================================
# Define held-out test datasets
# =============================================================================

TEST_FILES = {
    "Demographics + Questionnaire":
        PROCESSED_DIR
        / "test_demographics_questionnaire_task_aware.csv",

    "Wearable + Questionnaire":
        PROCESSED_DIR
        / "test_wearable_questionnaire_task_aware.csv",

    "Full Multimodal":
        PROCESSED_DIR
        / "test_multimodal_full_task_aware.csv",
}

In [10]:
# =============================================================================
# Verify required inputs
# =============================================================================

input_check_rows = []

for _, candidate in selected_candidates.iterrows():

    dataset = candidate["dataset"]
    model_name = candidate["model"]

    model_path = MODEL_FILES.get(
        (
            dataset,
            model_name,
        )
    )

    test_path = TEST_FILES.get(
        dataset
    )

    input_check_rows.append(
        {
            "Dataset": dataset,
            "Model": model_name,

            "Model File":
                model_path.name
                if model_path is not None
                else None,

            "Model Available":
                model_path.exists()
                if model_path is not None
                else False,

            "Test File":
                test_path.name
                if test_path is not None
                else None,

            "Test Available":
                test_path.exists()
                if test_path is not None
                else False,
        }
    )

input_check = pd.DataFrame(
    input_check_rows
)

display(input_check)

,Dataset,Model,Model File,Model Available,Test File,Test Available
0,Demographics + Questionnaire,Logistic Regression,demographics_plus_questionnaire_logistic_regre...,True,test_demographics_questionnaire_task_aware.csv,True
1,Full Multimodal,XGBoost,full_multimodal_xgboost.joblib,True,test_multimodal_full_task_aware.csv,True
2,Wearable + Questionnaire,Random Forest,wearable_plus_questionnaire_random_forest.joblib,True,test_wearable_questionnaire_task_aware.csv,True


In [11]:
assert input_check[
    "Model Available"
].all(), (
    "One or more selected model artifacts are missing."
)

assert input_check[
    "Test Available"
].all(), (
    "One or more held-out test datasets are missing."
)

print(
    "All final test inputs verified"
)

All final test inputs verified


#### 4. Verify Test Dataset Structure

In [12]:
# ===============================================================
# Verify held-out test dataset structure
# ===============================================================

test_structure_rows = []

for dataset, test_file in TEST_FILES.items():

    test_df = pd.read_csv(
        test_file,
        dtype={
            ID_COLUMN: str,
        },
    )

    if ID_COLUMN not in test_df.columns:
        raise ValueError(
            f"{ID_COLUMN!r} missing from {test_file.name}."
        )

    if TARGET not in test_df.columns:
        raise ValueError(
            f"{TARGET!r} missing from {test_file.name}."
        )

    duplicated_ids = (
        test_df[
            ID_COLUMN
        ]
        .duplicated()
        .sum()
    )

    missing_ids = (
        test_df[
            ID_COLUMN
        ]
        .isna()
        .sum()
    )

    missing_targets = (
        test_df[
            TARGET
        ]
        .isna()
        .sum()
    )

    test_structure_rows.append(
        {
            "Dataset":
                dataset,

            "Rows":
                len(test_df),

            "Unique Participants":
                test_df[
                    ID_COLUMN
                ].nunique(),

            "Duplicated Participants":
                duplicated_ids,

            "Missing Participant IDs":
                missing_ids,

            "Missing Targets":
                missing_targets,

            "Total Columns":
                test_df.shape[1],
        }
    )


test_structure_check = pd.DataFrame(
    test_structure_rows
)

display(test_structure_check)

,Dataset,Rows,Unique Participants,Duplicated Participants,Missing Participant IDs,Missing Targets,Total Columns
0,Demographics + Questionnaire,71,71,0,0,0,58
1,Wearable + Questionnaire,71,71,0,0,0,2402
2,Full Multimodal,71,71,0,0,0,2412


In [13]:
assert (
    test_structure_check[
        "Duplicated Participants"
    ]
    == 0
).all(), (
    "Duplicated participants detected "
    "in one or more test datasets."
)

assert (
    test_structure_check[
        "Missing Participant IDs"
    ]
    == 0
).all(), (
    "Missing participant IDs detected."
)

assert (
    test_structure_check[
        "Missing Targets"
    ]
    == 0
).all(), (
    "Missing target labels detected."
)

print(
    "Held-out Test datasets verified"
)

Held-out Test datasets verified


#### 5. Evaluate Selected Models on Held-Out Test Data

In [14]:
# =============================================================================
# Allowed missing feature for Week 5 pipeline compatibility
# =============================================================================

ALLOWED_MISSING_FEATURES = {"urinary_count",}

In [15]:
# =============================================================================
# Evaluate validation-selected models on held-out test data
# =============================================================================

test_results_rows = []
test_prediction_rows = []
test_probability_rows = []

classification_report_store = {}
confusion_matrix_store = {}

for _, candidate in selected_candidates.iterrows():

    dataset = candidate["dataset"]
    model_name = candidate["model"]

    print("\n" + "=" * 80)
    print(f"{dataset} | {model_name}")
    print("=" * 80)

    # -------------------------------------------------------------------------
    # Load fitted Week 5 pipeline
    # -------------------------------------------------------------------------

    model_path = MODEL_FILES[
        (
            dataset,
            model_name,
        )
    ]

    pipeline = joblib.load(
        model_path
    )

    # -------------------------------------------------------------------------
    # Load held-out test dataset
    # -------------------------------------------------------------------------

    test_file = TEST_FILES[
        dataset
    ]

    test_df = pd.read_csv(
        test_file,
        dtype={
            ID_COLUMN: str,
        },
    )

    y_test = (
        test_df[
            TARGET
        ]
        .copy()
    )

    # -------------------------------------------------------------------------
    # Recover the exact feature schema used in Week 5
    # -------------------------------------------------------------------------

    if hasattr(
        pipeline,
        "feature_names_in_",
    ):

        expected_columns = list(
            pipeline.feature_names_in_
        )

    else:

        expected_columns = [
            column
            for column in test_df.columns
            if column not in {
                TARGET,
                ID_COLUMN,
            }
        ]

    # -------------------------------------------------------------------------
    # Check missing features
    # -------------------------------------------------------------------------

    missing_columns = [
        column
        for column in expected_columns
        if column not in test_df.columns
    ]

    unsupported_missing = [
        column
        for column in missing_columns
        if column not in ALLOWED_MISSING_FEATURES
    ]

    if unsupported_missing:

        raise ValueError(
            "Test data are missing "
            f"{len(unsupported_missing)} "
            "unsupported feature(s): "
            f"{unsupported_missing[:10]}"
        )

    added_missing = []

    for column in missing_columns:

        test_df[
            column
        ] = np.nan

        added_missing.append(
            column
        )

    # -------------------------------------------------------------------------
    # Align test predictors to fitted pipeline
    # -------------------------------------------------------------------------

    X_test = (
        test_df[
            expected_columns
        ]
        .copy()
    )

    print(
        "Model expects:",
        len(expected_columns),
        "raw features"
    )

    print(
        "Test aligned:",
        X_test.shape[1],
        "raw features"
    )

    if added_missing:

        print(
            "Compatibility handling:",
            added_missing,
            "supplied as NaN",
        )

    # -------------------------------------------------------------------------
    # Final test prediction only
    # NO fit(), retuning, or feature-selection changes
    # -------------------------------------------------------------------------

    y_pred = pipeline.predict(
        X_test
    )

    # -------------------------------------------------------------------------
    # Shared evaluation utilities
    # -------------------------------------------------------------------------

    metrics = calculate_metrics(
        y_test,
        y_pred,
    )

    classification_report_df = (
        get_classification_report(
            y_test,
            y_pred,
        )
    )

    confusion_df = (
        get_confusion_matrix(
            y_test,
            y_pred,
        )
    )

    # -------------------------------------------------------------------------
    # Store model-level metrics
    # -------------------------------------------------------------------------

    test_results_rows.append(
        {
            "dataset":
                dataset,

            "model":
                model_name,

            "artifact_filename":
                model_path.name,

            "test_file":
                test_file.name,

            "n_test":
                len(y_test),

            "added_missing_features":
                ",".join(
                    added_missing
                )
                if added_missing
                else "",

            **metrics,
        }
    )

    result_key = (
        f"{dataset}__{model_name}"
    )

    classification_report_store[
        result_key
    ] = classification_report_df

    confusion_matrix_store[
        result_key
    ] = confusion_df

    # -------------------------------------------------------------------------
    # Store participant-level predictions
    # -------------------------------------------------------------------------

    participant_ids = (
        test_df[
            ID_COLUMN
        ]
        .astype(str)
        .values
    )

    for (
        participant_id,
        true_label,
        predicted_label,
    ) in zip(
        participant_ids,
        y_test,
        y_pred,
    ):

        test_prediction_rows.append(
            {
                ID_COLUMN:
                    participant_id,

                "dataset":
                    dataset,

                "model":
                    model_name,

                "true_label":
                    true_label,

                "predicted_label":
                    predicted_label,
            }
        )

    # -------------------------------------------------------------------------
    # Store predicted probabilities
    # -------------------------------------------------------------------------

    if hasattr(
        pipeline,
        "predict_proba",
    ):

        probabilities = (
            pipeline.predict_proba(
                X_test
            )
        )

        classes = getattr(
            pipeline,
            "classes_",
            None,
        )

        if (
            classes is None
            and hasattr(
                pipeline,
                "named_steps",
            )
        ):

            classifier = (
                pipeline
                .named_steps
                .get(
                    "classifier"
                )
            )

            classes = getattr(
                classifier,
                "classes_",
                np.arange(
                    probabilities.shape[1]
                ),
            )

        if classes is None:

            classes = np.arange(
                probabilities.shape[1]
            )

        for row_index, (
            participant_id,
            true_label,
        ) in enumerate(
            zip(
                participant_ids,
                y_test,
            )
        ):

            probability_row = {
                ID_COLUMN:
                    participant_id,

                "dataset":
                    dataset,

                "model":
                    model_name,

                "true_label":
                    true_label,
            }

            for (
                class_index,
                class_label,
            ) in enumerate(
                classes
            ):

                probability_row[
                    f"prob_class_{class_label}"
                ] = probabilities[
                    row_index,
                    class_index,
                ]

            test_probability_rows.append(
                probability_row
            )

    print(
        f"✓ Final test successful | "
        f"Macro F1={metrics['macro_f1']:.4f}"
    )


Demographics + Questionnaire | Logistic Regression
Model expects: 51 raw features
Test aligned: 51 raw features
✓ Final test successful | Macro F1=0.5750

Full Multimodal | XGBoost
Model expects: 2405 raw features
Test aligned: 2405 raw features
✓ Final test successful | Macro F1=0.6425

Wearable + Questionnaire | Random Forest
Model expects: 2397 raw features
Test aligned: 2397 raw features
✓ Final test successful | Macro F1=0.6157


#### 6. Final Test Results

The final performance metrics, participant-level predictions, and predicted class probabilities are consolidated and saved for the three validation-selected candidate models.

In [16]:
# =============================================================================
# Build final test result tables
# =============================================================================

test_results = pd.DataFrame(
    test_results_rows
)

test_predictions = pd.DataFrame(
    test_prediction_rows
)

test_probabilities = pd.DataFrame(
    test_probability_rows
)

display(
    test_results[
        [
            "dataset",
            "model",
            "n_test",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "precision_macro",
            "recall_macro",
        ]
    ].round(4)
)

,dataset,model,n_test,accuracy,balanced_accuracy,macro_f1,precision_macro,recall_macro
0,Demographics + Questionnaire,Logistic Regression,71,0.6338,0.6057,0.5750,0.5632,0.6057
1,Full Multimodal,XGBoost,71,0.7183,0.6172,0.6425,0.6911,0.6172
2,Wearable + Questionnaire,Random Forest,71,0.6620,0.6333,0.6157,0.6042,0.6333


#### Observation: 

The three validation-selected models were successfully evaluated on the held-out test dataset (71 participants). 

Among them, the Full Multimodal XGBoost achieved the highest overall accuracy (0.718), Macro F1-score (0.643), and macro precision (0.691). The Wearable + Questionnaire Random Forest achieved a lower accuracy (0.662) and Macro F1-score (0.616), but obtained the highest balanced accuracy (0.633 versus 0.617), while the Demographics + Questionnaire Logistic Regression showed the lowest performance.

This means the Full Multimodal XGBoost provided the strongest overall balances across the primary evaluation metrics.

In [17]:
# =============================================================================
# Final test output paths
# =============================================================================

TEST_RESULTS_OUTPUT = (
    METRICS_DIR
    / "test_results.csv"
)

TEST_PREDICTIONS_OUTPUT = (
    METRICS_DIR
    / "test_predictions.csv"
)

TEST_PROBABILITIES_OUTPUT = (
    METRICS_DIR
    / "test_probabilities.csv"
)

In [18]:
# =============================================================================
# Save final test results
# =============================================================================

test_results.to_csv(
    TEST_RESULTS_OUTPUT,
    index=False,
)

test_predictions.to_csv(
    TEST_PREDICTIONS_OUTPUT,
    index=False,
)

if not test_probabilities.empty:

    test_probabilities.to_csv(
        TEST_PROBABILITIES_OUTPUT,
        index=False,
    )

print(
    "Saved:",
    TEST_RESULTS_OUTPUT,
)

print(
    "Saved:",
    TEST_PREDICTIONS_OUTPUT,
)

if not test_probabilities.empty:

    print(
        "Saved:",
        TEST_PROBABILITIES_OUTPUT,
    )

Saved: C:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\outputs\metrics\test_results.csv
Saved: C:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\outputs\metrics\test_predictions.csv
Saved: C:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\outputs\metrics\test_probabilities.csv


#### 7. Final Classification Reports and Confusion Matrices

In [19]:
# ====================================================
# Create safe filenames
# ====================================================

def clean_text(value):
    """Convert text to a safe lowercase filename component."""

    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(value)
        .strip()
        .lower(),
    ).strip("_")

In [30]:
# =============================================================================
# Save classification reports and confusion matrices
# =============================================================================

saved_classification_reports = []
saved_confusion_matrices = []

for result_key in classification_report_store:

    dataset, model_name = (
        result_key.split(
            "__",
            maxsplit=1,
        )
    )

    safe_name = (
        f"{clean_text(dataset)}"
        f"_{clean_text(model_name)}"
    )

    # ---------------------------------------------------
    # Classification report
    # ---------------------------------------------------

    classification_report_df = (
        classification_report_store[
            result_key
        ]
    )

    report_path = (
        METRICS_DIR
        / (
            f"{safe_name}"
            "_test_classification_report.csv"
        )
    )

    classification_report_df.to_csv(
        report_path,
        index=True,
    )

    saved_classification_reports.append(
        report_path
    )

    # -----------------------------------------------
    # Confusion matrix
    # -----------------------------------------------

    confusion_df = (
        confusion_matrix_store[
            result_key
        ]
    )

    print(
        "\n",
        dataset,
        "|",
        model_name,
    )

    display(
        confusion_df
    )

    confusion_path = (
        METRICS_DIR
        / (
            f"{safe_name}"
            "_test_confusion_matrix.csv"
        )
    )

    confusion_df.to_csv(
        confusion_path,
        index=True,
    )
        # Save confusion matrix figure
    
    fig, ax = plt.subplots(figsize=(6, 5))

    im = ax.imshow(
        confusion_df.values,
        cmap="Blues",
    )

    ax.set_xticks(
        range(len(confusion_df.columns))
    )

    ax.set_xticklabels(
        confusion_df.columns,
        rotation=20,
    )

    ax.set_yticks(
        range(len(confusion_df.index))
    )

    ax.set_yticklabels(
        confusion_df.index,
    )

    ax.set_xlabel("Predicted Class")
    ax.set_ylabel("True Class")
    ax.set_title(
        f"{dataset} - {model_name}"
    )

    for i in range(confusion_df.shape[0]):
        for j in range(confusion_df.shape[1]):
            ax.text(
                j,
                i,
                confusion_df.iloc[i, j],
                ha="center",
                va="center",
                color="black",
                fontsize=11,
            )

    plt.colorbar(im)

    plt.tight_layout()

    figure_path = (
        FIGURES_DIR
        / (
            f"{safe_name}"
            "_test_confusion_matrix.png"
        )
    )

    plt.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)
    saved_confusion_matrices.append(
        confusion_path
    )
print(
    "\nClassification reports saved:",
    len(saved_classification_reports),
)
print(
    "Confusion matrices saved:",
    len(saved_confusion_matrices),
)



 Demographics + Questionnaire | Logistic Regression


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,9,2,1
True_PD,4,30,8
True_Other,5,6,6



 Full Multimodal | XGBoost


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,6,5,1
True_PD,2,37,3
True_Other,1,8,8



 Wearable + Questionnaire | Random Forest


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,9,2,1
True_PD,3,31,8
True_Other,3,7,7



Classification reports saved: 3
Confusion matrices saved: 3


#### Observation: 

The confusion matrices show differences in class-level performance across the three models. Parkinson's Disease (PD) was the best-classified group overall, particularly by the Full Multimodal XGBoost. 

The Full Multimodal XGBoost correctly classified 6 of 12 Healthy participants, 37 of 42 PD participants, and 8 of 17 Other participants. 

The Wearable + Questionnaire Random Forest correctly classified 9 of 12 Healthy participants, 31 of 42 PD participants, and 7 of 17 Other participants, while the Demographics + Questionnaire Logistic Regression correctly classified 9 of 12 Healthy participants, 30 of 42 PD participants, and 6 of 17 Other participants. 

Overall, the Full Multimodal XGBoost correctly classified the largest number of PD participants, while the other two models performed better for Healthy participants. However, the Other class remained more difficult to distinguish across the models.

#### 8. Validation vs. Test Performance

The final test performance of each validation-selected candidate is compared with its corresponding validation performance.

In [22]:
# ===========================================================
# Build validation vs. test comparison
# ===========================================================

comparison_rows = []

for _, candidate in selected_candidates.iterrows():

    dataset = candidate[
        "dataset"
    ]

    model_name = candidate[
        "model"
    ]

    validation_row = (
        validation_results[
            (
                validation_results[
                    "dataset"
                ]
                == dataset
            )
            & (
                validation_results[
                    "model"
                ]
                == model_name
            )
        ]
        .iloc[0]
    )

    test_row = (
        test_results[
            (
                test_results[
                    "dataset"
                ]
                == dataset
            )
            & (
                test_results[
                    "model"
                ]
                == model_name
            )
        ]
        .iloc[0]
    )

    comparison_rows.append(
        {
            "dataset":
                dataset,

            "model":
                model_name,

            "validation_accuracy":
                validation_row[
                    "accuracy"
                ],

            "test_accuracy":
                test_row[
                    "accuracy"
                ],

            "validation_balanced_accuracy":
                validation_row[
                    "balanced_accuracy"
                ],

            "test_balanced_accuracy":
                test_row[
                    "balanced_accuracy"
                ],

            "validation_macro_f1":
                validation_row[
                    "macro_f1"
                ],

            "test_macro_f1":
                test_row[
                    "macro_f1"
                ],

            "validation_precision_macro":
                validation_row[
                    "precision_macro"
                ],

            "test_precision_macro":
                test_row[
                    "precision_macro"
                ],

            "validation_recall_macro":
                validation_row[
                    "recall_macro"
                ],

            "test_recall_macro":
                test_row[
                    "recall_macro"
                ],
        }
    )

validation_test_comparison = pd.DataFrame(
    comparison_rows
)

In [23]:
# =========================================================
# Calculate validation-to-test changes
# =========================================================

validation_test_comparison[
    "accuracy_change"
] = (
    validation_test_comparison[
        "test_accuracy"
    ]
    - validation_test_comparison[
        "validation_accuracy"
    ]
)

validation_test_comparison[
    "balanced_accuracy_change"
] = (
    validation_test_comparison[
        "test_balanced_accuracy"
    ]
    - validation_test_comparison[
        "validation_balanced_accuracy"
    ]
)

validation_test_comparison[
    "macro_f1_change"
] = (
    validation_test_comparison[
        "test_macro_f1"
    ]
    - validation_test_comparison[
        "validation_macro_f1"
    ]
)

In [24]:
display(
    validation_test_comparison[
        [
            "dataset",
            "model",

            "validation_macro_f1",
            "test_macro_f1",
            "macro_f1_change",

            "validation_balanced_accuracy",
            "test_balanced_accuracy",
            "balanced_accuracy_change",

            "validation_accuracy",
            "test_accuracy",
            "accuracy_change",
        ]
    ].round(4)
)

,dataset,model,validation_macro_f1,test_macro_f1,macro_f1_change,validation_balanced_accuracy,test_balanced_accuracy,balanced_accuracy_change,validation_accuracy,test_accuracy,accuracy_change
0,Demographics + Questionnaire,Logistic Regression,0.6329,0.5750,-0.0579,0.7016,0.6057,-0.0958,0.6571,0.6338,-0.0233
1,Full Multimodal,XGBoost,0.7071,0.6425,-0.0646,0.6995,0.6172,-0.0823,0.7714,0.7183,-0.0531
2,Wearable + Questionnaire,Random Forest,0.7650,0.6157,-0.1494,0.7862,0.6333,-0.1529,0.8000,0.6620,-0.1380


In [25]:
# =============================================================================
# Save validation vs. test comparison
# =============================================================================

VALIDATION_TEST_COMPARISON_OUTPUT = (
    TABLES_DIR
    / "validation_test_comparison.csv"
)

validation_test_comparison.to_csv(
    VALIDATION_TEST_COMPARISON_OUTPUT,
    index=False,
)

print(
    "Saved:",
    VALIDATION_TEST_COMPARISON_OUTPUT,
)

Saved: C:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\outputs\tables\validation_test_comparison.csv


#### 9. Final Performance Summary

The held-out test results are summarized for the three models previously selected during validation.

In [26]:
# =============================================================================
# Final performance summary
# =============================================================================

final_performance_summary = (
    test_results[
        [
            "dataset",
            "model",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "precision_macro",
            "recall_macro",
        ]
    ]
    .copy()
)

display(
    final_performance_summary.round(4)
)

,dataset,model,accuracy,balanced_accuracy,macro_f1,precision_macro,recall_macro
0,Demographics + Questionnaire,Logistic Regression,0.6338,0.6057,0.5750,0.5632,0.6057
1,Full Multimodal,XGBoost,0.7183,0.6172,0.6425,0.6911,0.6172
2,Wearable + Questionnaire,Random Forest,0.6620,0.6333,0.6157,0.6042,0.6333


In [27]:
display(
    validation_test_comparison[
        [
            "dataset",
            "model",
            "macro_f1_change",
            "balanced_accuracy_change",
            "accuracy_change",
        ]
    ].round(4)
)

,dataset,model,macro_f1_change,balanced_accuracy_change,accuracy_change
0,Demographics + Questionnaire,Logistic Regression,-0.0579,-0.0958,-0.0233
1,Full Multimodal,XGBoost,-0.0646,-0.0823,-0.0531
2,Wearable + Questionnaire,Random Forest,-0.1494,-0.1529,-0.1380


In [28]:
print(
    "Final held-out test evaluation completed "
    "for all validation-selected candidates."
)

print(
    "No fitting, retuning, feature selection, "
    "or test-based model selection was performed."
)

Final held-out test evaluation completed for all validation-selected candidates.
No fitting, retuning, feature selection, or test-based model selection was performed.


#### 10. Deliverables Check

In [29]:
# ======================================================
# Verify final test deliverables
# ======================================================

deliverables = [
    {
        "Deliverable":
            "Final test results",

        "Path":
            TEST_RESULTS_OUTPUT,
    },

    {
        "Deliverable":
            "Final test predictions",

        "Path":
            TEST_PREDICTIONS_OUTPUT,
    },

    {
        "Deliverable":
            "Validation vs. test comparison",

        "Path":
            VALIDATION_TEST_COMPARISON_OUTPUT,
    },
]

if not test_probabilities.empty:

    deliverables.append(
        {
            "Deliverable":
                "Final test probabilities",

            "Path":
                TEST_PROBABILITIES_OUTPUT,
        }
    )


deliverables_df = pd.DataFrame(
    deliverables
)

deliverables_df[
    "Status"
] = (
    deliverables_df[
        "Path"
    ]
    .apply(
        lambda path: (
            "READY"
            if (
                Path(path).exists()
                and Path(path).stat().st_size > 0
            )
            else "MISSING"
        )
    )
)

display(
    deliverables_df
)

,Deliverable,Path,Status
0,Final test results,C:\Users\Daniela\Documents\UNF\Summer 2026-Ter...,READY
1,Final test predictions,C:\Users\Daniela\Documents\UNF\Summer 2026-Ter...,READY
2,Validation vs. test comparison,C:\Users\Daniela\Documents\UNF\Summer 2026-Ter...,READY
3,Final test probabilities,C:\Users\Daniela\Documents\UNF\Summer 2026-Ter...,READY
